In [5]:
import matplotlib.pyplot as plt
import numpy as np

class MLP:
    def __init__(self, layer_sizes, activation='tanh'):
        if activation not in ('sigmoid', 'relu', 'tanh'):
            raise ValueError(f"Unknown activation {activation}")
        self.layer_sizes = layer_sizes
        self.activation = activation
        self.W, self.b = [], []
        self.Z_list, self.A_list = [], []
        self.dW_list = [np.zeros((layer_sizes[i], layer_sizes[i+1]))
                        for i in range(len(layer_sizes)-1)]
        self.db_list = [np.zeros((1, layer_sizes[i+1]))
                        for i in range(len(layer_sizes)-1)]

        for i in range(len(layer_sizes) - 1):
            in_dim, out_dim = layer_sizes[i], layer_sizes[i+1]
            self.W.append(np.random.randn(in_dim, out_dim) * 0.1)
            self.b.append(np.zeros((1, out_dim)))

    def _activate(self, Z):
        if self.activation == 'sigmoid':
            return 1 / (1 + np.exp(-Z))
        elif self.activation == 'relu':
            return np.maximum(0, Z)
        elif self.activation == 'tanh':
            return np.tanh(Z)

    def _dactivate(self, Z):
        if self.activation == 'sigmoid':
            s = 1 / (1 + np.exp(-Z))
            return s * (1 - s)
        elif self.activation == 'relu':
            return (Z > 0).astype(float)
        elif self.activation == 'tanh':
            return 1 - np.tanh(Z) ** 2

    def forward(self, Z):
        self.Z_list, self.A_list = [], []
        A = Z
        self.A_list.append(A)
        for i in range(len(self.W)):
            Z = A.dot(self.W[i]) + self.b[i]
            self.Z_list.append(Z)
            A = self._activate(Z) if i < len(self.W) - 1 else Z
            self.A_list.append(A)
        return A

    def mse(self, y_pred, y_true):
        return np.mean((y_pred - y_true) ** 2)

    def backward(self, y_true):
        m = y_true.shape[0]
        # выходной слой (линейный выход + MSE)
        aL = self.A_list[-1]
        delta = 2 * (aL - y_true) / m
        a_prev = self.A_list[-2]
        dW = a_prev.T.dot(delta)
        db = np.sum(delta, axis=0, keepdims=True)
        self.dW_list, self.db_list = [dW], [db]
        # скрытые слои
        for l in range(len(self.layer_sizes) - 2, 0, -1):
            z = self.Z_list[l - 1]
            a_prev = self.A_list[l - 1]
            W_next = self.W[l]
            delta = delta.dot(W_next.T) * self._dactivate(z)
            dW = a_prev.T.dot(delta)
            db = np.sum(delta, axis=0, keepdims=True)
            self.dW_list.insert(0, dW)
            self.db_list.insert(0, db)

    def update_params(self, lr):
        for i in range(len(self.W)):
            self.W[i] -= lr * self.dW_list[i]
            self.b[i] -= lr * self.db_list[i]
            

In [6]:
model = MLP([1, 1], activation='tanh')
lr, epochs = 0.1, 10

loss_history = []
grad_norms = []

for epoch in range(1, epochs + 1):
    preds = model.forward(X)
    loss = model.mse(preds, Y)
    loss_history.append(loss)

    model.backward(Y)
    
    grads = np.concatenate(
        [g.ravel() for g in model.dW_list] +
        [g.ravel() for g in model.db_list]
    )

    # 2. Вычислите евклидову норму вектора grads
    norm = np.linalg.norm(grads)
    grad_norms.append(norm)

    model.update_params(lr)

    if epoch % 2 == 0:
        print(f"Epoch {epoch:>2d}: loss={loss:.4f}, ||grad||={grad_norms[-1]:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(loss_history, color='tab:blue')
ax1.set(title="Batch GD: Loss vs эпоха", xlabel="Эпоха", ylabel="Loss")
ax1.grid(True)

ax2.plot(grad_norms, color='tab:orange')
ax2.set(title="Batch GD: ||grad|| vs эпоха", xlabel="Эпоха", ylabel="||grad||")
ax2.grid(True)

plt.tight_layout()
plt.show() 

NameError: name 'Y' is not defined